# 去嵌入

这是一个关于如何使用 scikit-rf 执行去嵌入的简短教程。我们将首先介绍去嵌入的概念以及为什么需要它，并给出一个简单的用例场景。接下来，我们将介绍存在哪些类型的去嵌入，以及如何为您的应用选择合适的方法。最后，我们将看到如何编写代码以快速对 S 参数数据集执行去嵌入操作。

## 什么是去嵌入，它与校准有何不同？

让我们从一个例子开始。考虑对硅晶圆上构建的集成射频晶体管的测量，这些晶体管对于集成电路设计的紧凑模型开发至关重要。这种场景的典型测量设置涉及使用射频探针，其一端具有同轴连接器，另一端具有地-信号-地（GSG）探针尖端。该探针尖端落在晶圆上构建的 GSG 焊盘上，提供对被测晶体管的访问。要测量射频晶体管，实际被测器件（DUT）通过金属互连连接到晶体管三个端子中的两个的 GSG 焊盘，而第三个端子连接到射频地。在 FET 的共源极测量配置中，这意味着栅极和漏极端子连接到射频探针，源极端子接地。现在，测量晶体管端子电压的函数 S 参数。

但是这个晶体管 S 参数测量的校准参考平面在哪里？电缆和探针过渡的影响可以通过执行标准校准（如直通-反射-线（TRL）或短路-开路-负载-直通（SOLT））来消除，以将校准参考移动到探针尖端，或者换句话说，移动到晶圆上实现的 GSG 焊盘。阻抗标准基板（ISS）的使用提供了一组明确定义的校准标准，可用于为测量建立这样的参考平面。然而，在访问我们实际想要的射频晶体管的真实端子之前，仍然需要移除涉及金属互连的相当大的"测试夹具"。**去嵌入是指移除测试夹具对被测器件（DUT）测量的无关影响。**下图（从[这里](https://mos-ak.org/shanghai_2016/presentations/Bertrand_Ardouin_MOS-AK_Shanghai_2016.pdf)获取）显示了在晶圆上测量应用中通常需要移除的"测试夹具"示例。

<img src="figures/onwafer_pads.png" alt="onwafer-pads" width="500">

去嵌入过程与校准的不同之处在于，没有使用众所周知的标准件，要么是因为它们在某些环境中无法存在，要么是由于空间和成本限制而不实用。去嵌入使用几个虚拟结构来帮助移除不需要的测试夹具效应，但没有提供足够的信息来推导使用标准校准技术获得的完整误差盒网络。由于测试夹具本身在到达 DUT 之前可能包含各种过渡和互连，因此当参考平面的简单标量端口扩展不适用时，去嵌入非常有用。

有关此主题的基本介绍，请参阅[这篇文章](http://na.support.keysight.com/faq/deembed.pdf)。

## 开路-短路去嵌入

在过去几十年的晶圆上射频测量中，开路-短路去嵌入方法一直是射频集成电路行业的主力，其中晶体管的工作频率主要在几吉赫兹范围内。随着晶体管测量扩展到更高频率，当开路-短路去嵌入的简化假设失效时，需要更复杂的去嵌入方法。很难定义开路-短路去嵌入有效的频率上限，因为它取决于晶圆上焊盘笼设计中采用的布局技术。如果采用适当的设计技术，开路-短路去嵌入应至少适用于 10 GHz（如果不是更高）。

开路-短路去嵌入的准确性取决于测试夹具寄生元件是并联电导（红色）和串联阻抗（橙色）组合的假设的有效性，如下图所示。集总元件模型通常代表晶圆上测试夹具，因为首先是 GSG 焊盘中的并联焊盘电容，然后是互连线的串联阻抗。要将测量的参考平面从网络下方的外端子移动到 DUT，除了 DUT 之外，还需要两个虚拟结构 - 开路和短路。要创建开路虚拟结构，只需从测试夹具中移除 DUT，而将三个端子短接在一起以实现短路虚拟结构。借助这些虚拟结构，可以提取 DUT 的真实测量值。

<img src="figures/openshort.png" alt="openshort" width="500">

开路-短路去嵌入的详细过程如下图所示。开路虚拟结构的测量结果显然与并联寄生元件相似。并联寄生元件简单地通过测量开路虚拟结构来提取（步骤 i）。另一方面，短路虚拟结构包括并联和串联寄生元件。串联寄生元件通过在 Y 参数域中从短路虚拟结构中减去并联寄生元件来提取（步骤 ii）。通过在 Y 和 Z 参数域中分别从 DUT 测量结果中减去并联和串联寄生元件，可以提取没有寄生元件的 DUT（步骤 iii）。

<img src="figures/OpenShortProcedure.svg" alt="OpenShortProcedure" width="700">

读者可以参考[这个详细的演示](https://mos-ak.org/shanghai_2016/presentations/Bertrand_Ardouin_MOS-AK_Shanghai_2016.pdf)了解有关晶圆上测试结构设计的复杂细节和最佳实践。从这里开始，我们将重点解释 scikit-rf 中去嵌入类的工作原理。

## 使用 Scikit-RF 进行去嵌入

在本节中，让我们构建一个具体的示例来演示 scikit-rf 中的去嵌入是如何工作的。假设被测器件是一个 1nH 电感器，我们对其测量感兴趣。由于该电感器必须放置在晶圆上测试夹具中，让我们假设每个端口的焊盘电容为 25fF，焊盘间电容为 10fF，每个焊盘的互连线电阻为 2 欧姆。在外部参考平面 P1-P2 上可用的测量结果网络如下图所示。

<img src="figures/ind_parasitics.png" alt="openshort-ckt" width="500">

目标是准确提取实际的 1nH 电感器，通过移除所有其他无关的寄生电路元件。



In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import skrf as rf

In [ ]:
# Look at the raw inductor measurement with parasitics included
# From S11/S22, it is clear that it is not a pure inductance.
raw_ind = rf.data.ind
raw_ind.plot_s_smith()

如果我们要直接从该测量中提取电感，我们将得到一个随频率变化的电感值，品质因数也会受到寄生电阻存在的影响。

In [ ]:
# plot the inductance and q-factor of the raw inductor measurement
Lraw_nH = 1e9 * np.imag(1/raw_ind.y[:,0,0])/2/np.pi/raw_ind.f
Qraw = np.abs(np.imag(1/raw_ind.y[:,0,0])/np.real(1/raw_ind.y[:,0,0]))

fig, (ax1, ax2) = plt.subplots(1,2)
ax1.plot(raw_ind.f*1e-9, Lraw_nH)
ax1.grid()
ax1.set_ylabel("Inductance (nH)")
ax1.set_xlabel("Freq. (GHz)")
ax2.plot(raw_ind.f*1e-9, Qraw)
ax2.grid()
ax2.set_ylabel("Q-factor")
ax2.set_xlabel("Freq. (GHz)")
fig.tight_layout()

要创建开路虚拟结构，只需从 DUT 测试结构中移除 DUT（即 1nH 电感器），得到如下图所示的电路。

<img src="figures/ind_open.png" alt="openckt" width="500">

要创建短路虚拟结构，将测试夹具的内部端子（否则将连接到 DUT）短接到地，如下图所示。

<img src="figures/ind_short.png" alt="shortckt" width="500">

In [ ]:
# load in 2-ports short/open dummy networks
open_nw = rf.data.open_2p
short_nw = rf.data.short_2p

要使用可用的虚拟测量执行开路-短路去嵌入，我们将去嵌入对象创建为 `OpenShort` 类的实例，同时将开路和短路网络对象作为参数提供给去嵌入对象创建。

要获取去嵌入网络，我们在要执行去嵌入操作的网络对象上应用 `deembed` 方法。如下所示。

In [ ]:
from skrf.calibration.deembedding import OpenShort

dm = OpenShort(dummy_open=open_nw, dummy_short=short_nw, name='tutorial')

actual_ind = dm.deembed(raw_ind)
actual_ind.plot_s_smith()

In [ ]:
# plot the inductance of the de-embedded measurement
# we ignore plotting Q-factor here, because an ideal lossless inductor has infinite Q

Lactual_nH = 1e9 * np.imag(1/actual_ind.y[:,0,0])/2/np.pi/actual_ind.f

fig, ax1 = plt.subplots(1,1)
ax1.plot(actual_ind.f*1e-9, Lactual_nH)
ax1.grid()
ax1.set_ylim(0.95, 1.1)
ax1.set_ylabel("Inductance (nH)")
ax1.set_xlabel("Freq. (GHz)")
fig.tight_layout()

从上面的图中可以看出，即使在存在不想要的寄生元件的情况下，由于测试夹具的正确去嵌入，实际电感值也能被准确提取。

## 仅直通去嵌入

在 10 GHz 以上的更高频率下，开路虚拟结构的边缘电容和短路虚拟结构的寄生电感会引起误差。对于高频，已经提出了几种使用直通（thru）虚拟结构的方法，例如 `SplitPi`[[1](#ref1)]、`SplitTee`[[2](#ref2)]、`AdmittanceCancel`[[3](#ref3)] 和 `ImpedanceCancel`[[4](#ref4)]。直通虚拟结构是左右测试焊盘的直接连接。测试器件和直通虚拟结构的示例如下图所示，其中 G 和 S 分别表示地焊盘和信号焊盘。

<img src="figures/thru_micrograph.png" alt="thru_micrograph" width="500">

### 分离方法

分离方法基于集总元件假设以及开路或短路方法。`SplitPi` 和 `SplitTee` 方法假设嵌入网络分别是 Π 网络和 T 网络，如下图所示。左右寄生元件（即 Z 和 Y）的值从直通虚拟结构中提取。寄生元件通过乘以左右 ABCD 矩阵的逆来移除。**这些方法仅适用于 2 端口 DUT**。

<img src="figures/split_methods.svg" alt="split_methods" width="500">

### 导纳抵消方法

`AdmittanceCancel`（又称 Mangan 方法）和 `ImpedanceCancel` 方法也基于集总元件假设，如下图所示。然而，与分离方法不同，它们不直接计算左右寄生元件的导纳。这些方法通过称为"交换"的左右镜像操作来去嵌入 DUT。`AdmittanceCancel` 方法包括两个步骤。首先，通过将测试器件的 ABCD 矩阵乘以直通虚拟结构的逆矩阵来计算定义为'H'的矩阵，如图(c)所示。然后，通过取 H 的左右镜像和非镜像 Y 参数的平均值来获得去嵌入的 Y 矩阵。类似地，在 `ImpedanceCancel` 方法的情况下，它取 Z 参数的平均值。**这些方法仅适用于对称的 2 端口 DUT（即 S11=S22 和 S12=S21）**，但在毫米波频率下传输线表征中比分离方法更准确。分离方法和导纳抵消方法之间的比较在 [[4]](https://ieeexplore.ieee.org/document/7377897) 中描述。

<img src="figures/immittance_cancel.svg" alt="immittance_cancel" width="500">

## 其他去嵌入方法

scikit-rf 中包含的其他简单形式的去嵌入有 `Open`、`Short` 和 `ShortOpen` 方法，根据测试夹具寄生网络的等效电路，这些方法可能适用。例如，可能希望仅从测量中移除焊盘接触电阻，为此可以使用 `Short` 去嵌入。在某些测量中，可能只需要移除焊盘电容，为此 `Open` 去嵌入方法更合适。要一次性移除焊盘接触电阻和焊盘电容，`ShortOpen` 方法是最合适的。

文献中还报道了许多其他复杂的去嵌入方法，以将 DUT 测量的精度扩展到高频。虽然可以使用 scikit-rf 的网络操作来完成这些方法，但将它们作为内置去嵌入类包含在包中是受欢迎的开源贡献。

## 参考文献

<div id="ref1"></div>[1]: [L. Nan et al., "Experimental Characterization of the Effect of Metal Dummy Fills on Spiral Inductors," in 2007 IEEE Radio Frequency Integrated Circuits (RFIC) Symposium, Jun. 2007, pp. 307–310.](https://ieeexplore.ieee.org/document/4266437)

<div id="ref2"></div>[2]: [M. J. Kobrinsky et al., "Experimental validation of crosstalk simulations for on-chip interconnects using S-parameters," IEEE Transactions on Advanced Packaging, vol. 28, no. 1, pp. 57–62, Feb. 2005.](https://ieeexplore.ieee.org/document/1391067)

<div id="ref3"></div>[3]: [A. M. Mangan et al., "De-embedding transmission line measurements for accurate modeling of IC designs," IEEE Trans. Electron Devices, vol. 53, no. 2, pp. 235–241, Feb. 2006.](https://ieeexplore.ieee.org/document/1580859)

<div id="ref4"></div>[4]: [S. Amakawa et al., "Comparative analysis of on-chip transmission line de-embedding techniques," in 2015 IEEE International Symposium on Radio-Frequency Integration Technology, Aug. 2015, pp. 91–93.](https://ieeexplore.ieee.org/document/7377897)